# Your agent doesn't know when to stop

*Build a retrieve-judge-retry loop, watch it rescue a question that failed, then measure it against doing nothing clever.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/flows/03_when_to_stop_searching.ipynb)
[![Open in GitHub Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** A retrieve-judge-retry loop can cost four times as much as answering once and buy nothing measurable, while the boring alternative — retrieve more text, answer once — does better for free.
**Result.** In the run stored here: the loop rescues a governing-law clause the single-shot pipeline missed, then scores 65.6% across 160 judged questions against 68.1% for the pipeline it was built to beat — 11 questions fixed, 15 broken, for 705 model calls against 160. Retrieving five times as much text and answering once scores 73.8% on the same 160 calls. Rebuilding the stopping rule so that absence must be proven does not rescue it, because the loop spends its extra reading on exactly the questions that have no answer: 2.2× more of the contract than it reads for the questions that do.
**Requires.** couchbase · llm · local-embeddings
**Read** ~14 min · **Run** ~32 min · **Cost** a dollar or two — about 1,700 model calls, most of them reading whole contracts

[`flows/01`](01_rag_that_you_can_trust.ipynb) left a specific failure on the table. Of the
questions whose answer a lawyer had marked in the contract, retrieval simply did not fetch the
marked passage for a third of them. The model never had a chance.

There is an obvious thing to do about that, and it is the thing everyone is building right now:
**don't answer on the first try.** Retrieve, look at what came back, and if it isn't good
enough, search again with better words. A loop. An agent.

This notebook builds that loop, and it works — on the question that motivated it. Then it runs
the loop over the same 160 judged questions `flows/01` used, against the most boring
alternative available: retrieving more text and answering once.

The corpus is [CUAD](https://www.atticusprojectai.org/cuad) again, and it is the right corpus
for this question because of one property:

> **62% of the questions have no answer in their contract.**

A loop that searches until it finds something has to decide when to give up. On a corpus where
giving up is usually the correct answer, that decision is not a detail. It is the whole system.

**What this notebook does**

1. Finds a question the single-shot pipeline gets wrong, and watches it fail.
2. Builds a loop that searches again, and watches it fix that question.
3. Runs both over 160 judged questions, against doing nothing clever.
4. Follows every question to find where the answers went.
5. Rebuilds the stopping rule, and measures that instead.
6. Works out why looking harder did not help.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it. Once: a
        # restarted session keeps the files but not sys.path.
        root = pathlib.Path("cbnb-repo").resolve()
        if not root.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, str(root)])
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "llm", "local-embeddings"], extras=["plots"])

## 1. A clause that is in the contract, and a pipeline that says it isn't

The index is the one `flows/01` built: the same twenty contracts, split into overlapping
1,200-character chunks, embedded and stored in Couchbase. If it is already there this is a
no-op; if you have not run `flows/01`, it is built here.

In [2]:
# Load the contracts and split them into overlapping chunks, as flows/01 did.
import pandas as pd

from cbnb.datasets import load_cuad

CHUNK_CHARS, CHUNK_OVERLAP = 1200, 200

cuad = load_cuad()
questions = cuad.questions_with_absences()


def chunk(text, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    """Fixed-width overlapping windows, each carrying its character range."""
    step = size - overlap
    return [
        {"start": start, "end": min(start + size, len(text)), "text": text[start:start + size]}
        for start in range(0, max(len(text) - overlap, 1), step)
    ]


chunks = pd.DataFrame([
    {"chunk_id": f"{row.contract_id}-{number}", "contract_id": row.contract_id,
     "contract_type": row.contract_type, **piece}
    for row in cuad.contracts.itertuples(index=False)
    for number, piece in enumerate(chunk(row.text))
])

# The ground truth: which chunks overlap a span a lawyer marked.
gold = {}
for span in cuad.spans.itertuples(index=False):
    same = chunks[chunks.contract_id == span.contract_id]
    hits = same[(same.start < span.end) & (same.end > span.start)]
    gold.setdefault((span.contract_id, span.category), set()).update(hits.chunk_id)

print(f"{len(chunks):,} chunks from {len(cuad.contracts)} contracts, "
      f"{len(questions)} question/contract pairs "
      f"({(~questions.answerable).mean():.0%} with no answer)")

609 chunks from 20 contracts, 820 question/contract pairs (62% with no answer)


In [3]:
# Embed the chunks and index them in Couchbase. Idempotent: flows/01 built this.
from cbnb.couchbase_io import (connect, ensure_collection, ensure_vector_index, upsert_docs,
                               wait_for_index)
from cbnb.embeddings import Embedder

BUCKET, SCOPE, COLLECTION, INDEX = settings.cb_bucket, "contracts", "chunks", "contract_chunks"

embedder = Embedder()
cluster = connect(settings)
collection = ensure_collection(cluster, BUCKET, SCOPE, COLLECTION)

vectors = embedder.encode(chunks.text.tolist())
upsert_docs(collection, {
    row.chunk_id: {
        "type": COLLECTION, "chunk_id": row.chunk_id, "contract_id": int(row.contract_id),
        "contract_key": f"c{row.contract_id}", "contract_type": row.contract_type,
        "start": int(row.start), "end": int(row.end), "text": row.text,
        "embedding": vector.tolist(),
    }
    for row, vector in zip(chunks.itertuples(index=False), vectors)
})
ensure_vector_index(
    cluster, bucket_name=BUCKET, scope_name=SCOPE, collection_name=COLLECTION,
    index_name=INDEX, vector_field="embedding", dims=embedder.dims,
    text_fields=["text"], keyword_fields=["contract_key", "contract_type"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
               expected=len(chunks))

  upserted 200/609

  upserted 400/609

  upserted 600/609

  upserted 609/609

  indexed 609/609 (ready)   

  indexed 609/609 (ready)   

609

Retrieval and answering are `flows/01`'s, unchanged. The model must return a verbatim quote,
which is what lets the grading below be a string comparison against a lawyer's annotation
rather than a second model's opinion.

In [4]:
# Retrieve from one contract, then answer from the excerpts. flows/01's pipeline.
import couchbase.search as search
from pydantic import BaseModel, Field

from cbnb.couchbase_io import vector_search
from cbnb.llm import LLM

llm = LLM()
lookup = cuad.categories.set_index("category").question


class Answer(BaseModel):
    found: bool = Field(description="true only if the contract text below answers the question")
    quote: str = Field(description="the exact sentence from the text that supports the answer, copied verbatim; empty if not found")
    answer: str = Field(description="a one-sentence answer, or why the contract does not say")


SYSTEM = """You answer questions about a contract using only the excerpts provided.
Quote verbatim from the excerpts. If they do not answer the question, say so."""


def retrieve(contract_id, query, k):
    return vector_search(
        cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
        vector_field="embedding", query_vector=embedder.encode_one(query),
        k=k, num_candidates=100, fields=["text"],
        prefilter=search.TermQuery(f"c{contract_id}", field="contract_key"),
    )


def ask(contract_id, query, question, k):
    """One retrieval, one answer. Returns what came back and what was read."""
    hits = retrieve(contract_id, query, k)
    excerpts = "\n\n---\n\n".join(hit["text"] for hit in hits)
    reply = llm.structured(
        f"Contract excerpts:\n\n{excerpts}\n\nQuestion: {question}",
        Answer, system=SYSTEM, max_tokens=2000,
    )
    return {"ids": [hit.id for hit in hits], "found": reply.found, "quote": reply.quote,
            "answer": reply.answer, "k": k, "query": query,
            # Working data for the policy below; the stored trace drops it.
            "_excerpts": [hit["text"] for hit in hits]}

Here is one of the failures, and it is not a subtle one. Every commercial contract says which
state's law governs it. This pipeline cannot find it.

In [5]:
# Ask one question whose answer is certainly in the contract, and watch the pipeline miss it.
K_BASE = 4
FAILING = (14, "Governing Law")

question = lookup[FAILING[1]]
first = ask(FAILING[0], question, question, K_BASE)

print(f"contract c{FAILING[0]} - {FAILING[1]}")
print(f"question: {question}\n")
print(f"found:  {first['found']}")
print(f"answer: {first['answer']}\n")
marked = gold[FAILING]
print(f"a lawyer marked {len(marked)} of this contract's chunks as holding the answer")
print(f"we retrieved {len(set(first['ids']) & marked)} of them")

contract c14 - Governing Law
question: Which state/country's law governs the interpretation of the contract?

found:  False
answer: The excerpt's Governing Law clause (Section 5.6) is cut off mid-sentence ('This Agreement a...') and does not specify which state or country's law governs the contract.

a lawyer marked 2 of this contract's chunks as holding the answer
we retrieved 1 of them


Read the model's own explanation before reading on, because it is more specific than "retrieval
failed". The right part of the contract *was* retrieved — and the 1,200-character window cut
the clause in half, so the sentence naming the state never made it into the excerpt. The
pipeline is not confused. It is answering honestly about text that was truncated before it
arrived.

## 2. The obvious fix: look again, with better words

The question failed because the retrieval query was the question itself, and a question is not
phrased the way a contract is. *"Is there a minimum order size?"* has to find text that says
*minimum purchase commitment*. One fix is to let a model read what came back and write a better
query — and if that is worth doing once, it is worth doing in a loop until the answer is good.

That loop is the thing this notebook is about, so it is worth being precise about its parts.
A loop like this is a **harness**: a step, a budget, a stopping rule and a record of what
happened, with a *policy* inside it that decides what to do next. The harness is in
[`cbnb.agent`](../../cbnb/agent.py) because it is bookkeeping. The policy stays here, because
deciding when a result is good enough is the technique — and, it turns out, the whole story.

Two adjustments are available:

| action | what changes |
| --- | --- |
| `rephrase` | a new retrieval query, written by the policy in the contract's vocabulary |
| `widen` | the same query, more chunks — `k` moves up through 4, 10, 20 |

And one rule that is the agentic bet in a sentence: **never accept the first "not found."**
If the model says the clause isn't there on the first try, look again. That is what would have
rescued the question above, and it is stated here plainly because everything that follows is
a measurement of whether it was a good idea.

In [6]:
# The policy: read what came back, decide whether to stop or what to change.
from cbnb.agent import Budget, Move, run

WIDENING = [4, 10, 20]


class Control(BaseModel):
    accept: bool = Field(description="true if this answer is final")
    next_action: str = Field(description="one of: widen, rephrase, none")
    new_query: str = Field(description="a different retrieval query, if rephrasing; else empty")
    why: str = Field(description="one short sentence")


CONTROLLER = """You are checking whether a search of one contract is finished.
Accept a well-supported answer. Also accept "not in this contract" - but only once the
excerpts show you have looked in the right part of the document, never on the first try.
If it is not final, choose widen (same query, more excerpts) or rephrase (a different
retrieval query, using the words a contract would actually use)."""


def state_text(state):
    """What the policy is shown: the excerpts, the verdict, and what has been tried."""
    obs = state.observation
    excerpts = "\n".join(f"  {n + 1}. {text[:200]}..."
                         for n, text in enumerate(obs["_excerpts"][:5]))
    tried = "\n".join(f'  "{s.action["query"]}" (k={s.action["k"]}) -> '
                      f'{"found" if s.observation["found"] else "nothing"}'
                      for s in state.history)
    return (f"Question: {state.task}\n"
            f'Retrieval query used: "{obs["query"]}" (k={obs["k"]})\n'
            f"Excerpts retrieved:\n{excerpts or '  (nothing retrieved)'}\n"
            f"The answering model said: found={obs['found']}, "
            f"quote: \"{obs['quote'][:300]}\"\n"
            f"Already tried:\n{tried}\n"
            f"Adjustments left: {state.steps_left}")


def model_stops(state):
    """Ask the model whether to stop. Never accept the first 'not found'."""
    reply = llm.structured(state_text(state), Control, system=CONTROLLER, max_tokens=2000)
    evidence = reply.model_dump()
    forced = not state.observation["found"] and len(state.history) == 1

    if (reply.accept or reply.next_action == "none") and not forced:
        return Move(stop=True, reason=reply.why, evidence=evidence)

    if reply.next_action == "rephrase" and reply.new_query.strip():
        action = {"kind": "rephrase", "query": reply.new_query.strip(),
                  "k": state.observation["k"]}
    else:
        wider = next((w for w in WIDENING if w > state.observation["k"]), None)
        if wider is None:
            return Move(stop=True, reason="nothing wider left", evidence=evidence)
        action = {"kind": "widen", "query": state.observation["query"], "k": wider}
    return Move(stop=False, action=action, reason=reply.why, evidence=evidence)

The budget caps the loop at three steps. It also *could* cap wall-clock time, and for a
production agent it should — but a clock makes the result depend on how the network was
feeling, and rewrites the trajectories this notebook is going to read. Steps are the only
bound that is reproducible, so steps are the only bound used here.

In [7]:
# Run one task to a stop. The harness handles the loop; the policy decides.
def search_until_done(contract_id, category, policy, budget=None):
    question = lookup[category]

    def act(action):
        return ask(contract_id, action["query"], question, action["k"])

    return run(
        f"c{contract_id}::{category}", question,
        start={"kind": "retrieve", "query": question, "k": WIDENING[0]},
        act=act, policy=policy,
        budget=budget or Budget(max_steps=3, max_seconds=float("inf")),
        contract_id=int(contract_id), category=category,
    )

## 3. On the question that motivated it, it works

Same question, same corpus, same answering model. The only difference is that the pipeline is
allowed to have a second thought.

In [8]:
# Run the loop on the question the single-shot pipeline missed.
rescued = search_until_done(FAILING[0], FAILING[1], model_stops)

for step in rescued.steps:
    verdict = "found it" if step.observation["found"] else "nothing"
    print(f"step {step.n}: k={step.action['k']:2d}  {verdict}")
    print(f"         query: {step.observation['query'][:88]}")
    if step.move:
        print(f"         -> {step.move['reason'][:88]}")
print(f"\nstopped: {rescued.stopped}")
print(f"the lawyer's chunk was {'' if set(rescued.final_ids) & gold[FAILING] else 'not '}"
      f"retrieved by the end")

step 1: k= 4  nothing
         query: Which state/country's law governs the interpretation of the contract?
         -> The excerpts don't cover the governing law clause; rephrase using standard contract lang
step 2: k= 4  found it
         query: governing law jurisdiction venue this agreement shall be construed in accordance with th
         -> Excerpt 1 shows the Governing Law; Jurisdiction section (5.6) and the quote specifies Ma

stopped: satisfied
the lawyer's chunk was retrieved by the end


That is the demo. It is a good demo: a real failure, a plausible fix, and the fix works in
front of you. `flows/01` opened the same way, and the lesson was the same — one question is
not evidence.

## 4. Now ask all 160

The same sample `flows/01` used, graded the same deterministic way: the model's quote either
lands inside a span a lawyer marked, or it does not. Five outcomes, no judge.

- **grounded** — answered, and the quote is inside a marked span.
- **abstained** — said "not in this contract", and it genuinely is not. *Correct.*
- **wrong span** — answered, but quoted something the lawyer did not mark.
- **missed** — said "not in this contract", but it is.
- **hallucinated** — answered a question that has no answer.

Three of the five rows below are not loops at all. They exist because a loop that costs four
times as much has to beat the cheapest thing anyone would try instead, and the cheapest thing
is to skip the cleverness and retrieve more text.

In [9]:
# Grade one result against the lawyers' spans: five outcomes, no judge.
def grade(row, found, quote):
    spans = cuad.spans[(cuad.spans.contract_id == row.contract_id)
                       & (cuad.spans.category == row.category)]
    if not row.answerable:
        return "abstained" if not found else "hallucinated"
    if not found:
        return "missed"
    said = " ".join(quote.split())
    for span in spans.itertuples(index=False):
        marked = " ".join(span.text.split())
        if said and (said in marked or marked in said):
            return "grounded"
    return "wrong span"


N_EVAL = 160
sample = questions.sample(N_EVAL, random_state=1729).reset_index(drop=True)
print(f"{len(sample)} questions: {sample.answerable.sum()} answerable, "
      f"{(~sample.answerable).sum()} with no answer")

160 questions: 67 answerable, 93 with no answer


In [10]:
# Run every strategy over the sample, in parallel. Answers are cached; re-running is free.
from concurrent.futures import ThreadPoolExecutor


def scored(row, found, quote, ids, steps, answer_calls, control_calls, stopped):
    want = gold.get((row.contract_id, row.category), set())
    return {"contract_id": row.contract_id, "category": row.category,
            "answerable": bool(row.answerable), "steps": steps, "stopped": stopped,
            "retrieved_gold": any(i in want for i in ids) if want else None,
            "answer_calls": answer_calls, "control_calls": control_calls,
            "outcome": grade(row, found, quote)}


def single_shot(row, k):
    question = lookup[row.category]
    got = ask(row.contract_id, question, question, k)
    return scored(row, got["found"], got["quote"], got["ids"], 1, 1, 0, "single-shot")


def looped(row, policy, budget=None):
    trace = search_until_done(row.contract_id, row.category, policy, budget)
    last = trace.steps[-1].observation
    # A policy that asked a model left its answer in evidence; one that only
    # counted chunks left it empty. Only the first kind costs a call.
    controls = sum(1 for step in trace.steps if step.move and step.move["evidence"])
    return trace, scored(row, last["found"], last["quote"], trace.final_ids,
                         len(trace.steps), len(trace.steps), controls, trace.stopped)


def over_sample(fn):
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(fn, sample.itertuples(index=False)))

In [11]:
# Two single shots: the pipeline as flows/01 left it, and the same with more context.
rows_k4 = pd.DataFrame(over_sample(lambda r: single_shot(r, 4)))
rows_k20 = pd.DataFrame(over_sample(lambda r: single_shot(r, 20)))
print(f"{len(rows_k4)} questions asked twice, at k=4 and k=20")

160 questions asked twice, at k=4 and k=20


In [12]:
# The loop, over the same questions. Four times the calls, and it takes about as long.
agentic = over_sample(lambda r: looped(r, model_stops))
traces = [trace for trace, _ in agentic]
rows_loop = pd.DataFrame([row for _, row in agentic])
print(f"{rows_loop.steps.sum()} steps over {len(rows_loop)} questions, "
      f"plus {rows_loop.control_calls.sum()} calls asking whether to stop")

408 steps over 160 questions, plus 297 calls asking whether to stop


In [13]:
# Tally the outcomes, the cost, and how many steps each strategy took.
def summarise(frame, name):
    counts = frame.outcome.value_counts()
    correct = counts.get("grounded", 0) + counts.get("abstained", 0)
    return pd.Series({
        "grounded": counts.get("grounded", 0), "abstained": counts.get("abstained", 0),
        "wrong span": counts.get("wrong span", 0), "missed": counts.get("missed", 0),
        "hallucinated": counts.get("hallucinated", 0),
        "correct %": 100 * correct / len(frame),
        "model calls": frame.answer_calls.sum() + frame.control_calls.sum(),
        "mean steps": frame.steps.mean(),
    }, name=name)


# Counts are whole numbers; only the two derived columns want a decimal.
SHOWN = {name: "{:,.0f}" for name in
         ["grounded", "abstained", "wrong span", "missed", "hallucinated", "model calls"]}
SHOWN |= {"correct %": "{:.1f}", "mean steps": "{:.1f}"}

pd.DataFrame([summarise(rows_k4, "single-shot, k=4"),
              summarise(rows_k20, "single-shot, k=20"),
              summarise(rows_loop, "agentic, model stops")]).style.format(SHOWN)

,grounded,abstained,wrong span,missed,hallucinated,correct %,model calls,mean steps
"single-shot, k=4",34,75,13,20,18,68.1,160,1.0
"single-shot, k=20",52,66,8,7,27,73.8,160,1.0
"agentic, model stops",42,63,10,15,30,65.6,705,2.5


The loop does not win.

It lands a little behind the pipeline it was built to beat — 2.5 points on 160 questions, which
is within the noise — and it spent **705 model calls** to get there against that pipeline's 160.

The row that does win is **single-shot k=20**, which has no cleverness in it at all: retrieve
five times as much text, answer once, go home. Same 160 calls, eight points better than the
loop.

A percentage on 160 questions moves by a few points on its own, so the number to look at is not
the column but the disagreements — the questions where two strategies gave different verdicts.
Everything else is the same question answered the same way twice.

In [14]:
# A percentage on 160 questions moves on its own. Compare the pairs that disagree.
from math import comb


def versus(left, right, left_name, right_name):
    """Only the questions where two strategies disagree carry any signal."""
    ok = lambda frame: frame.outcome.isin(["grounded", "abstained"])   # noqa: E731
    fixed = int(((~ok(left)) & ok(right)).sum())
    broke = int((ok(left) & (~ok(right))).sum())
    n = fixed + broke
    # Exact two-sided sign test on the disagreements.
    smaller = min(fixed, broke)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(smaller + 1)) / 2 ** n) if n else 1.0
    return pd.Series({"fixed": fixed, "broke": broke, "disagreements": n, "p": p},
                     name=f"{right_name} vs {left_name}")


pd.DataFrame([versus(rows_k4, rows_loop, "single-shot k=4", "agentic"),
              versus(rows_k20, rows_loop, "single-shot k=20", "agentic"),
              versus(rows_k4, rows_k20, "single-shot k=4", "single-shot k=20")]
             ).style.format({"fixed": "{:.0f}", "broke": "{:.0f}",
                            "disagreements": "{:.0f}", "p": "{:.3f}"})

,fixed,broke,disagreements,p
agentic vs single-shot k=4,11,15,26,0.557
agentic vs single-shot k=20,7,20,27,0.019
single-shot k=20 vs single-shot k=4,20,11,31,0.150


## 5. Where the answers went

The aggregate says the loop is worse. It does not say what it did. Because every question was
asked both ways, each one can be followed from its single-shot outcome to its agentic one.

In [15]:
# Follow every question from its single-shot outcome to its agentic one.
pd.crosstab(rows_k4.outcome, rows_loop.outcome,
            rownames=["single-shot k=4"], colnames=["agentic"])

agentic,abstained,grounded,hallucinated,missed,wrong span
single-shot k=4,,,,,
abstained,61,0,14,0,0
grounded,0,33,0,1,0
hallucinated,2,0,16,0,0
missed,0,7,0,13,0
wrong span,0,2,0,1,10


Read the diagonal first — those are the questions the loop did not change, and most of them are
on it. Then read the row for **abstained**: the 75 questions the single-shot pipeline correctly
said were not in the contract. The loop held on to 61 of them, and turned **14 into
fabrications**.

Fourteen is not many. It is also nearly everything the loop broke — 14 of its 15 — and it is
spent in one direction: on questions that had already been answered correctly, sent back to look
again until something turned up.

In [16]:
# Watch one correct abstention become a hallucination, step by step.
became = [(t, s) for t, s in zip(traces, rows_loop.itertuples(index=False))
          if s.outcome == "hallucinated" and not s.answerable]
trace, _ = became[0]

print(f"{trace.task}\n")
print("this contract has no such clause; the lawyers marked nothing.\n")
for step in trace.steps:
    found = step.observation["found"]
    print(f"step {step.n}: k={step.action['k']:2d}  "
          f"{'FOUND' if found else 'nothing'}")
    print(f"         query: {step.observation['query'][:84]}")
    if found:
        print(f"         quote: {step.observation['quote'][:84]}")
    if step.move:
        print(f"         -> {step.move['reason'][:84]}")

Does the contract contain a license granted by one party to its counterparty?

this contract has no such clause; the lawyers marked nothing.

step 1: k= 4  nothing
         query: Does the contract contain a license granted by one party to its counterparty?
         -> The initial query was generic and excerpts show no license language; rephrase using 
step 2: k= 4  FOUND
         query: license grant trademark software intellectual property rights granted
         quote: Affiliate acknowledges that the use of the Network 1 name is on a non-exclusive basi
         -> The excerpt shows Network 1 grants Affiliate a non-exclusive license to use its name


Nothing in that trajectory looks like a malfunction. The reasoning is fluent, the actions are
sensible, each query is a reasonable thing to try next. Read the trace and the loop looks like
it is working. Only the ground truth says otherwise — which is the argument for having any.

## 6. Blame the stopping rule, and fix it

The loop's *actions* look fine. Rephrasing found clauses the first query missed — section 3
showed that, and the crosstab confirms it at scale. The suspect is the rule for stopping: it
treats **"I have not found it yet"** as **"keep looking"**, and on a corpus where most
questions have no answer, that is a machine for inventing them.

So replace it. And notice that the replacement does not need a better prompt — it needs a
better question. *"Have I looked everywhere yet?"* is not a judgement. It is a fact about the
corpus, and it can be counted: these contracts are 14 to 42 chunks long, so once the loop has
seen all of them, absence is **demonstrated** rather than guessed.

Note what that means. The decision that matters here needs no model at all.

In [17]:
# Stop when the answer is found, or when the whole contract has been read.
COVERAGE = [4, 10, 20, 40]
contract_size = chunks.groupby("contract_id").size().to_dict()


def coverage_stops(contract_id):
    """Absence is accepted only once every chunk of the contract has been seen."""
    def policy(state):
        seen = {i for step in state.history for i in step.result_ids}
        if state.observation["found"]:
            return Move(stop=True, reason="the answer is here")
        if len(seen) >= contract_size[contract_id]:
            return Move(stop=True, reason="the whole contract has been read; it is not here")
        wider = next((w for w in COVERAGE if w > state.observation["k"]), None)
        if wider is None:
            return Move(stop=True, reason="nothing wider left")
        return Move(stop=False, reason=f"{len(seen)} of {contract_size[contract_id]} chunks seen",
                    action={"kind": "widen", "query": state.observation["query"], "k": wider})
    return policy


print(f"chunks per contract: {min(contract_size.values())} to {max(contract_size.values())}")

chunks per contract: 14 to 42


In [18]:
# A single shot big enough to hold most whole contracts, for comparison.
rows_k40 = pd.DataFrame(over_sample(lambda r: single_shot(r, 40)))
print(f"k=40 covers the whole contract for "
      f"{sum(1 for n in contract_size.values() if n <= 40)} of {len(contract_size)} contracts")

k=40 covers the whole contract for 16 of 20 contracts


In [19]:
# The same loop, with only the stopping rule changed.
covered = over_sample(lambda r: looped(
    r, coverage_stops(r.contract_id), Budget(max_steps=len(COVERAGE), max_seconds=float("inf"))))
traces += [trace for trace, _ in covered]
rows_cov = pd.DataFrame([row for _, row in covered])
print(f"{rows_cov.steps.sum()} steps over {len(rows_cov)} questions, "
      f"plus {rows_cov.control_calls.sum()} calls asking whether to stop")

376 steps over 160 questions, plus 0 calls asking whether to stop


In [20]:
# Every strategy, side by side.
pd.DataFrame([summarise(rows_k4, "single-shot, k=4"),
              summarise(rows_k20, "single-shot, k=20"),
              summarise(rows_k40, "single-shot, k=40"),
              summarise(rows_loop, "agentic, model stops"),
              summarise(rows_cov, "agentic, coverage stops")]).style.format(SHOWN)

,grounded,abstained,wrong span,missed,hallucinated,correct %,model calls,mean steps
"single-shot, k=4",34,75,13,20,18,68.1,160,1.0
"single-shot, k=20",52,66,8,7,27,73.8,160,1.0
"single-shot, k=40",56,63,6,5,30,74.4,160,1.0
"agentic, model stops",42,63,10,15,30,65.6,705,2.5
"agentic, coverage stops",46,56,16,5,37,63.8,376,2.4


In [21]:
# Did changing only the stopping rule change the answer?
pd.DataFrame([versus(rows_loop, rows_cov, "model stops", "coverage stops"),
              versus(rows_k40, rows_cov, "single-shot k=40", "coverage stops")]
             ).style.format({"fixed": "{:.0f}", "broke": "{:.0f}",
                            "disagreements": "{:.0f}", "p": "{:.3f}"})

,fixed,broke,disagreements,p
coverage stops vs model stops,11,14,25,0.690
coverage stops vs single-shot k=40,1,18,19,0.000


It does not rescue the loop.

Against the model's own stopping rule it is a wash: 11 questions fixed, 14 broken, which is
what a coin does. Against a single k=40 retrieval it is behind: 18 questions broken against 1
fixed, which at p < 0.001 is not chance.

What it does do is produce **more fabrications than any other strategy here** — including the
loop it was supposed to repair, and including the flat retrieval that reads twice as much text.

So the stopping rule was not the problem, or at least not the whole of it. Something about
looking harder is wrong in a way that a better-designed loop does not fix.

## 7. Why more looking does not help

Set the loops aside and read the three single-shot rows again, as a series in how much text
the model was shown.

In [22]:
# What does an extra chunk of context buy, and what does it cost?
def by_answerable(frame, name):
    unanswerable, answerable = frame[~frame.answerable], frame[frame.answerable]
    return pd.Series({
        "grounded % (of answerable)":
            100 * (answerable.outcome == "grounded").mean(),
        "hallucinated % (of unanswerable)":
            100 * (unanswerable.outcome == "hallucinated").mean(),
    }, name=name)


pd.DataFrame([by_answerable(rows_k4, "single-shot, k=4"),
              by_answerable(rows_k20, "single-shot, k=20"),
              by_answerable(rows_k40, "single-shot, k=40"),
              by_answerable(rows_cov, "agentic, coverage stops")]).style.format("{:.1f}")

,grounded % (of answerable),hallucinated % (of unanswerable)
"single-shot, k=4",50.7,19.4
"single-shot, k=20",77.6,29.0
"single-shot, k=40",83.6,32.3
"agentic, coverage stops",68.7,39.8


Every extra chunk buys recall on the questions that have an answer and costs correct
abstentions on the questions that do not. That trade is not a bug in the loop — it is a
property of asking a language model to find something in a pile of text, and it holds for a
single shot with no loop anywhere near it.

Which explains the loops, once you ask **where each one spent its extra reading**.

In [23]:
# Where did each loop spend its extra context: on questions with answers, or without?
def spending(frame, name):
    return pd.Series({
        "chunks read, answerable": frame[frame.answerable].chunks_read.mean(),
        "chunks read, unanswerable": frame[~frame.answerable].chunks_read.mean(),
        "steps, answerable": frame[frame.answerable].steps.mean(),
        "steps, unanswerable": frame[~frame.answerable].steps.mean(),
    }, name=name)


for frame, these in ((rows_loop, traces[:N_EVAL]), (rows_cov, traces[N_EVAL:])):
    frame["chunks_read"] = [len({i for step in t.steps for i in step.result_ids})
                            for t in these]

table = pd.DataFrame([spending(rows_loop, "agentic, model stops"),
                      spending(rows_cov, "agentic, coverage stops")])
table["ratio"] = table["chunks read, unanswerable"] / table["chunks read, answerable"]
table.style.format("{:.2f}")

,"chunks read, answerable","chunks read, unanswerable","steps, answerable","steps, unanswerable",ratio
"agentic, model stops",8.04,7.41,2.24,2.77,0.92
"agentic, coverage stops",8.97,20.02,1.60,2.89,2.23


There it is.

The coverage loop reads **2.2× more of the contract on questions that have no answer**. Not by
accident — by construction. A question whose answer exists is found early and the loop stops
after about 1.6 steps; a question whose answer does not exist keeps returning "not found", so
the loop keeps widening, and by its third step it is asking a model to find a clause in an
entire contract that does not contain one.

It reads *least* where reading helps and *most* where reading hurts. That is how it manages to
be behind a flat k=40 on both counts at once — fewer grounded answers and more fabrications —
while reading only about a fifth as much text on the questions that have an answer.

The model-stopped loop sits at a ratio near 1.0. It does not aim its extra effort anywhere in
particular, so it simply pays four times as much for nothing.

## What to take from this

**An adaptive strategy is only as good as the correlation between its trigger and where the
extra effort pays off.** The trigger here is *"I did not find it"*, and on a corpus where 62%
of the questions have no answer, that is mostly evidence the clause is not there. So the loop
concentrates its spending precisely where spending is harmful. Flat retrieval is worse at
everything except the one thing that decided the result: it does not aim.

**More context is a trade, not a free improvement.** Going from 4 chunks to 40 took grounded
answers from 51% to 84% of the answerable questions — and fabrications from 19% to 32% of the
unanswerable ones. There is no setting of `k` that escapes that trade, and nothing about a loop
changes its shape.

**Stopping at the first "found" is sampling until success.** A loop that retries until
something turns up gives the model several independent chances to produce a false positive and
keeps the first one. The coverage loop reads about half as much of the contract as a flat k=40
on the questions with no answer, and still fabricates at least as often — which is hard to
explain by context size, and easy to explain by the number of attempts.

**The reasoning looked fine the whole time.** Read any trajectory above: the queries are
sensible, the justifications are fluent and specific, each step is a reasonable thing to try
next. Nothing in the traces announces a problem. The only thing that caught it was 160
questions with answers a lawyer wrote down.

**Ask what your system does when the answer is absent.** Every failure in this notebook lives
in the 62% of questions that have none. A benchmark where every question has an answer cannot
see any of it, and most benchmarks are built that way.

## Where to take this

- **Predict instead of reacting.** The trigger here fires *after* a failed search, which is
  the worst possible time — by then the evidence points at absence. Deciding up front whether
  a clause is likely present is a cheap typed classification, and
  [`flows/03`](03_a_cheaper_controller.ipynb) measures one that answers in about 200ms.
- **Measure the trajectories, not just the answers.** Every run above wrote a trace: what it
  did, why, and what came back at each step. [`flows/04`](04_reading_the_trajectories.ipynb)
  stores them in Couchbase and asks which steps were worth taking.
- **Try it where the document does not fit.** The largest contract here is 41,501 characters,
  so "read the whole thing" was always available. On 300-page agreements it is not, and a
  stopping rule has to be built out of something else — coverage of a *section*, or a
  retrieval score floor.
- **Ask what your corpus does when the answer is absent.** Every failure in this notebook is
  invisible on a benchmark where every question has an answer. Most benchmarks do.

## What produced these results

Every number above came from the models below. A different model, or the same name served
differently, will give different numbers.


In [24]:
# Name the models that produced the results above.
cbnb.run_card()

,model,how it ran
Chat model,z-ai/glm-5.3,"via NanoGPT; 1,576 calls"
Embeddings,sentence-transformers/all-MiniLM-L6-v2,"on this machine; 1,876 texts, 1,696 from cache"


In [25]:
# Optional clean-up: delete what this notebook created. Uncomment to run.
# It lives in the scope flows/01 created, which this notebook reuses.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)